In [ ]:
import os
import tkinter as tk

class GestorAmigos:
    ARCHIVO = "friendsContact.txt"

    @staticmethod
    def inicializar_archivo():
        if not os.path.exists(GestorAmigos.ARCHIVO):
            open(GestorAmigos.ARCHIVO, "w", encoding="utf-8").close()

    @staticmethod
    def agregar(nombre, numero):
        GestorAmigos.inicializar_archivo()

        with open(GestorAmigos.ARCHIVO, "r", encoding="utf-8") as f:
            for linea in f:
                if linea.strip():
                    partes = linea.strip().split("!")
                    if partes[0] == nombre or partes[1] == str(numero):
                        raise ValueError("El nombre o el número ya existen.")

        with open(GestorAmigos.ARCHIVO, "a", encoding="utf-8") as f:
            f.write(f"{nombre}!{numero}\n")

    @staticmethod
    def consultar_todos():
        GestorAmigos.inicializar_archivo()
        resultado = []
        with open(GestorAmigos.ARCHIVO, "r", encoding="utf-8") as f:
            for linea in f:
                if linea.strip():
                    partes = linea.strip().split("!")
                    resultado.append(f"Amigo: {partes[0]} | Tel: {partes[1]}")
        return "\n".join(resultado) if resultado else "(Archivo vacío)"

    @staticmethod
    def modificar(nombre_buscar, nuevo_numero):
        GestorAmigos.inicializar_archivo()
        encontrado = False
        lineas_nuevas = []

        # Leemos el archivo línea por línea
        with open(GestorAmigos.ARCHIVO, "r", encoding="utf-8") as f:
            for linea in f:
                if linea.strip():
                    partes = linea.strip().split("!")
                    if partes[0] == nombre_buscar:
                        lineas_nuevas.append(f"{partes[0]}!{nuevo_numero}\n")
                        encontrado = True
                    else:
                        lineas_nuevas.append(linea)

        if not encontrado:
            raise ValueError("El nombre ingresado no existe.")

        with open(GestorAmigos.ARCHIVO, "w", encoding="utf-8") as f:
            f.writelines(lineas_nuevas)

    @staticmethod
    def eliminar(nombre_buscar):
        GestorAmigos.inicializar_archivo()
        encontrado = False
        lineas_nuevas = []

        with open(GestorAmigos.ARCHIVO, "r", encoding="utf-8") as f:
            for linea in f:
                if linea.strip():
                    partes = linea.strip().split("!")
                    if partes[0] == nombre_buscar:
                        encontrado = True
                        continue
                    lineas_nuevas.append(linea)

        if not encontrado:
            raise ValueError("El nombre ingresado no existe.")

        with open(GestorAmigos.ARCHIVO, "w", encoding="utf-8") as f:
            f.writelines(lineas_nuevas)


class Ventana:
    def __init__(self):
        self.ventana = tk.Tk()
        self.ventana.title("Agenda de Amigos")
        self.ventana.geometry("500x600")

        tk.Label(self.ventana, text="GESTIÓN DE CONTACTOS", font=("Arial", 12, "bold")).pack(pady=10)

        self.bloque_datos = tk.Frame(self.ventana)
        self.bloque_datos.pack(pady=10)

        tk.Label(self.bloque_datos, text="Nombre:").grid(row=0, column=0, sticky="e", pady=5)
        self.entrada_nombre = tk.Entry(self.bloque_datos, width=25)
        self.entrada_nombre.grid(row=0, column=1, padx=10, pady=5)

        tk.Label(self.bloque_datos, text="Número (Cel):").grid(row=1, column=0, sticky="e", pady=5)
        self.entrada_numero = tk.Entry(self.bloque_datos, width=25)
        self.entrada_numero.grid(row=1, column=1, padx=10, pady=5)

        self.bloque_botones = tk.Frame(self.ventana)
        self.bloque_botones.pack(pady=10)

        tk.Button(self.bloque_botones, text="Ingresar", width=12, command=self.accion_ingresar).grid(row=0, column=0, padx=5, pady=5)
        tk.Button(self.bloque_botones, text="Consultar", width=12, command=self.accion_consultar).grid(row=0, column=1, padx=5, pady=5)
        tk.Button(self.bloque_botones, text="Modificar", width=12, command=self.accion_modificar).grid(row=1, column=0, padx=5, pady=5)
        tk.Button(self.bloque_botones, text="Eliminar", width=12, command=self.accion_eliminar).grid(row=1, column=1, padx=5, pady=5)

        tk.Label(self.ventana, text="Resultados de la agenda:", font=("Arial", 10, "bold")).pack(pady=5)

        self.label_consola = tk.Label(self.ventana, text="", font=("Arial", 10, "italic"), justify="left")
        self.label_consola.pack(pady=10, padx=20)

        self.ventana.mainloop()

    def limpiar_campos(self):
        self.entrada_nombre.delete(0, tk.END)
        self.entrada_numero.delete(0, tk.END)

    def accion_ingresar(self):
        nom = self.entrada_nombre.get().strip()
        num_str = self.entrada_numero.get().strip()

        if nom == "" or num_str == "":
            self.label_consola.config(text="Llena ambos campos para registrar.")
            return
        try:
            num = int(num_str)
            GestorAmigos.agregar(nom, num)
            self.label_consola.config(text=f"Contacto '{nom}' añadido correctamente.")
            self.limpiar_campos()
        except ValueError as e:
            if "invalid literal" in str(e):
                self.label_consola.config(text="¡Error! El número debe contener solo dígitos.")
            else:
                self.label_consola.config(text=f"¡Error! {e}")

    def accion_consultar(self):
        contactos = GestorAmigos.consultar_todos()
        self.label_consola.config(text=contactos, fg="black")

    def accion_modificar(self):
        nom = self.entrada_nombre.get().strip()
        num_str = self.entrada_numero.get().strip()

        if nom == "" or num_str == "":
            self.label_consola.config(text="Escribe el nombre y su número nuevo.")
            return
        try:
            num = int(num_str)
            GestorAmigos.modificar(nom, num)
            self.label_consola.config(text=f"Contacto de '{nom}' actualizado.")
            self.limpiar_campos()
        except ValueError as e:
            if "invalid literal" in str(e):
                self.label_consola.config(text="El número debe contener solo dígitos.")
            else:
                self.label_consola.config(text=f"Error {e}")

    def accion_eliminar(self):
        nom = self.entrada_nombre.get().strip()

        if nom == "":
            self.label_consola.config(text="Escribe el nombre del amigo a eliminar.")
            return
        try:
            GestorAmigos.eliminar(nom)
            self.label_consola.config(text=f"'{nom}' fue eliminado de la agenda.")
            self.limpiar_campos()
        except ValueError as e:
            self.label_consola.config(text=f"{e}")

app = Ventana()